# Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations nibabel scipy

# Imports + Global Config

In [ ]:
import os, cv2, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import nibabel as nib
from tqdm import tqdm
from scipy.spatial.distance import directed_hausdorff

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 5

# Dataset Selector

In [ ]:
DATASET = "isic"  # dataset names "isic","brats" or "synapse"

# Contrast Enhancement

In [ ]:
def apply_clahe(image):
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
    Works for RGB images
    """
    image = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(image)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)


def apply_hist_equalization(image):
    """
    Global histogram equalization
    """
    image = cv2.cvtColor(image, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = cv2.split(image)

    y_eq = cv2.equalizeHist(y)

    merged = cv2.merge((y_eq, cr, cb))
    return cv2.cvtColor(merged, cv2.COLOR_YCrCb2RGB)


def apply_gamma_correction(image, gamma=1.5):
    """
    Gamma correction
    gamma > 1 → darker
    gamma < 1 → brighter
    """
    invGamma = 1.0 / gamma
    table = np.array([
        ((i / 255.0) ** invGamma) * 255 for i in np.arange(256)
    ]).astype("uint8")

    return cv2.LUT(image, table)


# -------------------------------
# Albumentations Wrapper
# -------------------------------

class ContrastEnhancement(A.ImageOnlyTransform):
    def __init__(self, method="clahe", gamma=1.5, always_apply=False, p=1.0):
        super().__init__(always_apply, p)
        self.method = method
        self.gamma = gamma

    def apply(self, image, **params):
        if self.method == "clahe":
            return apply_clahe(image)
        elif self.method == "hist":
            return apply_hist_equalization(image)
        elif self.method == "gamma":
            return apply_gamma_correction(image, self.gamma)
        else:
            return image

# Dataset Classes

# ISIC (2D Images) # 

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, split="train"):
        base_path = "/kaggle/input/datasets/yatharthtripathi56/isic-2018-skin-lesion-segmentation/ISIC 2018 Skin Lesion Segmentation"

        self.img_dir = os.path.join(base_path, "images", split)
        self.mask_dir = os.path.join(base_path, "annotations", split)

        self.samples = []

        img_files = os.listdir(self.img_dir)

        for img_name in img_files:
            if not (img_name.endswith(".jpg") or img_name.endswith(".png")):
                continue

            img_path = os.path.join(self.img_dir, img_name)

            # mask naming convention
            base_name = img_name.split(".")[0]
            mask_name = base_name + "_segmentation.png"
            mask_path = os.path.join(self.mask_dir, mask_name)

            # Ensure both exist
            if not os.path.exists(mask_path):
                continue

            self.samples.append((img_path, mask_path))

        self.tf = A.Compose([
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            ContrastEnhancement(method="clahe", p=1.0),
            #ContrastEnhancement(method="hist", p=0.3),
            #ContrastEnhancement(method="gamma", gamma=1.2, p=0.3),
            A.HorizontalFlip(p=0.5),
            A.Normalize(),
            ToTensorV2()
        ])

        print(f"[{split}] Total samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        img_path, mask_path = self.samples[i]

        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)

        # Safety check
        if img is None:
            raise ValueError(f"Bad image: {img_path}")
        if mask is None:
            raise ValueError(f"Bad mask: {mask_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = (mask > 127).astype(np.float32)

        aug = self.tf(image=img, mask=mask)
        return aug["image"], aug["mask"].unsqueeze(0)

# Dataset Loader Selector

In [ ]:
def get_dataset():
        if DATASET == "isic":
            return ISICDataset(max_patients=10)
        else:
            raise ValueError(f"Unknown DATASET: {DATASET}")

# Model (U-Net)

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)

# GAN Refinement Module

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet("resnet18", in_channels=3, classes=1)

    def forward(self, x):
        return self.model(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 1, 4, 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

adv_loss = nn.BCELoss()

# Diffusion Refinement Module

In [ ]:
def add_noise(x, t=0.1):
    noise = torch.randn_like(x)
    return x + t * noise


def denoise(model, x, steps=3):
    for _ in range(steps):
        x = model(x)
    return x

 # Loss + Metrics #

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

def dice(pred, target):
    pred = (torch.sigmoid(pred) > 0.5).float()
    inter = (pred * target).sum()
    return (2*inter)/(pred.sum()+target.sum()+1e-7)

# Advanced Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        smooth = 1e-7
        inter = (pred * target).sum()
        return 1 - (2*inter + smooth)/(pred.sum()+target.sum()+smooth)


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


loss_fn = CombinedLoss()

In [ ]:
# -------------------------------
# Train / Val Split (MANDATORY)
# -------------------------------
train_dataset = ISICDataset(split="train")
val_dataset   = ISICDataset(split="val")

loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Temporary unlabeled (same as train)
unlabeled_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Semi-Supervised Learning (Pseudo-label + Consistency) training 

In [ ]:

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer_d = torch.optim.Adam(D.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    model.train()
    G.train()
    D.train()

    unlabeled_iter = iter(unlabeled_loader)

    for x_l, y_l in loader:
        x_l, y_l = x_l.to(DEVICE), y_l.to(DEVICE)

        # Get unlabeled batch safely
        try:
            x_u, _ = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            x_u, _ = next(unlabeled_iter)

        x_u = x_u.to(DEVICE)

        # -------------------------
        # Supervised Loss
        # -------------------------
        pred_l = model(x_l)
        loss_sup = loss_fn(pred_l, y_l)

        # -------------------------
        # Pseudo-label (soft, NOT thresholded)
        # -------------------------
        with torch.no_grad():
            pseudo = torch.sigmoid(model(x_u))

        pred_u = model(x_u)

        # Consistency loss
        loss_cons = F.mse_loss(torch.sigmoid(pred_u), pseudo)

        # -------------------------
        # Diffusion Refinement
        # -------------------------
        noisy = add_noise(x_l)
        refined = model(noisy)
        loss_diff = loss_fn(refined, y_l)

        # -------------------------
        # GAN Training
        # -------------------------
        fake_pred = G(x_l)

        # Discriminator outputs
        d_real = D(y_l)
        d_fake = D(fake_pred.detach())

        # Labels must match discriminator output shape
        real_label = torch.ones_like(d_real)
        fake_label = torch.zeros_like(d_fake)

        # Train Discriminator
        loss_d = adv_loss(d_real, real_label) + adv_loss(d_fake, fake_label)

        optimizer_d.zero_grad()
        loss_d.backward()
        optimizer_d.step()

        # Train Generator
        loss_g = adv_loss(D(fake_pred), real_label)

        # -------------------------
        # Total Loss
        # -------------------------
        total_loss = (
            loss_sup
            + 0.5 * loss_cons
            + 0.3 * loss_diff
            + 0.1 * loss_g
        )

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {total_loss.item():.4f}")

# Ablation / Comparative Analysis

In [ ]:
results = []

def run_experiment(name, use_gan=False, use_diff=False):
    score = np.random.rand()  # placeholder
    results.append((name, score))

run_experiment("baseline")
run_experiment("semi_supervised")
run_experiment("with_gan", use_gan=True)
run_experiment("with_diffusion", use_diff=True)

print(results)

In [ ]:
def metrics(pred, target):
    pred = (pred > 0.5).astype(np.float32)
    target = target.astype(np.float32)

    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    tn = ((1 - pred) * (1 - target)).sum()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-7)
    iou = tp / (tp + fp + fn + 1e-7)

    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    f1 = (2 * precision * recall) / (precision + recall + 1e-7)

    specificity = tn / (tn + fp + 1e-7)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-7)

    return [dice, iou, precision, recall, f1, specificity, accuracy]

# Evaluation

In [ ]:
model.eval()
scores = []

with torch.no_grad():
    for x, y in loader:
        x = x.to(DEVICE)

        pred = torch.sigmoid(model(x)).cpu().numpy()
        y = y.numpy()

        for p, t in zip(pred, y):
            scores.append(metrics(p[0], t[0]))

scores = np.array(scores)

print("Dice:", scores[:, 0].mean())
print("IoU:", scores[:, 1].mean())
print("Precision:", scores[:, 2].mean())
print("Recall:", scores[:, 3].mean())
print("F1-score:", scores[:, 4].mean())
print("Specificity:", scores[:, 5].mean())
print("Accuracy:", scores[:, 6].mean())

# Precision/Recall/F1 bar graph

In [ ]:
import matplotlib.pyplot as plt

metric_names = ["Precision", "Recall", "F1-score"]
metric_values = [
    scores[:, 2].mean(),
    scores[:, 3].mean(),
    scores[:, 4].mean()
]

plt.figure(figsize=(6, 4))
plt.bar(metric_names, metric_values)
plt.ylim(0, 1)
plt.title("Precision, Recall, and F1-score Comparison")
plt.ylabel("Score")
plt.show()

# Visualization / Graphs

In [ ]:
import matplotlib.pyplot as plt

# Example visualization
x_sample, y_sample = next(iter(loader))
x_sample = x_sample.to(DEVICE)

with torch.no_grad():
    pred = torch.sigmoid(model(x_sample)).cpu()

plt.figure(figsize=(10,5))

plt.subplot(1,3,1)
plt.imshow(x_sample[0].permute(1,2,0).cpu())
plt.title("Input")

plt.subplot(1,3,2)
plt.imshow(y_sample[0][0], cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred[0][0], cmap='gray')
plt.title("Prediction")

plt.show()

# Save Output

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/model.pth")